# Text Classification Pipeline
End-to-end pipeline for the News Category Dataset.

## 1. Environment and Dependencies

In [ ]:
%pip install -q scikit-learn pandas numpy matplotlib seaborn nltk wordcloud plotly joblib

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
print("NOTE: Notebook generated per user request")

## 2. Configuration

In [ ]:
import random
import numpy as np

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

FILENAME = "News_Category_Dataset_v3.json"
TEXT_COLUMN = "text"
LABEL_COLUMN = "category"
MAX_ROWS = 100000

## 3. Dataset Loading

In [ ]:
import os
import json
import pandas as pd

def load_dataset(filename):
    if not os.path.exists(filename):
        raise FileNotFoundError(f"File not found: {filename}")
    if filename.endswith(".csv"):
        df = pd.read_csv(filename)
    elif filename.endswith(".json") or filename.endswith(".jsonl"):
        records = []
        bad_lines = 0
        with open(filename, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError:
                    bad_lines += 1
                    continue
        if len(records) == 0:
            with open(filename, "r", encoding="utf-8") as f:
                content = f.read()
            try:
                records = json.loads(content)
            except json.JSONDecodeError as exc:
                raise ValueError(f"Could not parse file as JSON or JSONL: {exc}")
        df = pd.DataFrame.from_records(records)
        if bad_lines > 0:
            print(f"Skipped {bad_lines} malformed lines while loading")
    else:
        raise ValueError("Unsupported file format. Use CSV or JSON/JSONL.")
    return df

raw_df = load_dataset(FILENAME)
if raw_df.empty:
    raise ValueError("Loaded dataset is empty")

required_cols = ["headline", "category"]
for col in required_cols:
    if col not in raw_df.columns:
        raise KeyError(f"Required column missing from dataset: {col}")

if "short_description" in raw_df.columns:
    raw_df["text"] = raw_df["headline"].fillna("") + " " + raw_df["short_description"].fillna("")
else:
    raw_df["text"] = raw_df["headline"].fillna("")

raw_df = raw_df.rename(columns={"category": LABEL_COLUMN})
raw_df = raw_df[[TEXT_COLUMN, LABEL_COLUMN]].dropna()
raw_df = raw_df[raw_df[TEXT_COLUMN].str.strip().str.len() > 0]

if len(raw_df) > MAX_ROWS:
    raw_df = raw_df.sample(n=MAX_ROWS, random_state=RANDOM_STATE).reset_index(drop=True)

class_counts = raw_df[LABEL_COLUMN].value_counts()
print("Number of classes:", class_counts.shape[0])
print(class_counts)
raw_df.head(10)

## 4. Basic Dataset Exploration

In [ ]:
small_classes = class_counts[class_counts < 2]
if len(small_classes) > 0:
    raw_df = raw_df[~raw_df[LABEL_COLUMN].isin(small_classes.index)].reset_index(drop=True)
    class_counts = raw_df[LABEL_COLUMN].value_counts()

raw_df["word_count"] = raw_df[TEXT_COLUMN].apply(lambda x: len(str(x).split()))
raw_df["char_count"] = raw_df[TEXT_COLUMN].apply(lambda x: len(str(x)))

print("Average word count:", raw_df["word_count"].mean())
print("Median word count:", raw_df["word_count"].median())
print("Average char count:", raw_df["char_count"].mean())
print("Median char count:", raw_df["char_count"].median())

sample_classes = class_counts.index[:5]
for cls in sample_classes:
    subset = raw_df[raw_df[LABEL_COLUMN] == cls]
    longest = subset.loc[subset["word_count"].idxmax(), TEXT_COLUMN]
    shortest = subset.loc[subset["word_count"].idxmin(), TEXT_COLUMN]
    print("Class:", cls)
    print("Longest:", longest)
    print("Shortest:", shortest)
    print("---")

## 5. Text Preprocessing

In [ ]:
import re
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

STOPWORD_SET = set(stopwords.words("english"))
CUSTOM_STOPWORDS = set()
STOPWORD_SET = STOPWORD_SET.union(CUSTOM_STOPWORDS)
LEMMATIZER = WordNetLemmatizer()

def clean_text(text):
    text = str(text).lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\d+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def apply_preprocessing(df, text_column):
    df = df.copy()
    cleaned = df[text_column].apply(clean_text)
    tokens = cleaned.apply(word_tokenize)
    tokens = tokens.apply(lambda toks: [t for t in toks if t not in STOPWORD_SET and len(t) > 1])
    tokens = tokens.apply(lambda toks: [LEMMATIZER.lemmatize(t) for t in toks])
    df["tokens"] = tokens
    df["clean_text"] = tokens.apply(lambda toks: " ".join(toks))
    return df

processed_df = apply_preprocessing(raw_df, TEXT_COLUMN)
processed_df = processed_df[processed_df["clean_text"].str.strip().str.len() > 0].reset_index(drop=True)
processed_df.head(10)

## 6. Exploratory Text Analysis

In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

os.makedirs("output", exist_ok=True)

all_tokens = [tok for toks in processed_df["tokens"] for tok in toks]
top_overall = Counter(all_tokens).most_common(30)

plt.figure(figsize=(10, 8))
words, counts = zip(*top_overall)
sns.barplot(x=list(counts), y=list(words))
plt.title("Top 30 Most Common Words Overall")
plt.tight_layout()
plt.savefig("output/top_words_overall.png")
plt.show()

In [ ]:
top_classes = processed_df[LABEL_COLUMN].value_counts().index[:6]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
for i, cls in enumerate(top_classes):
    cls_tokens = [tok for toks in processed_df[processed_df[LABEL_COLUMN] == cls]["tokens"] for tok in toks]
    top_cls = Counter(cls_tokens).most_common(20)
    if len(top_cls) == 0:
        continue
    words, counts = zip(*top_cls)
    sns.barplot(x=list(counts), y=list(words), ax=axes[i])
    axes[i].set_title(f"Top Words: {cls}")
plt.tight_layout()
plt.savefig("output/top_words_per_class.png")
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
sns.countplot(y=processed_df[LABEL_COLUMN], order=processed_df[LABEL_COLUMN].value_counts().index)
plt.title("Class Distribution")
plt.tight_layout()
plt.savefig("output/class_distribution.png")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(processed_df["word_count"], bins=50)
plt.title("Text Length Distribution (Words)")
plt.tight_layout()
plt.savefig("output/text_length_distribution.png")
plt.show()

In [ ]:
from wordcloud import WordCloud

os.makedirs("output/wordclouds", exist_ok=True)
for cls in top_classes:
    text_blob = " ".join(processed_df[processed_df[LABEL_COLUMN] == cls]["clean_text"])
    if len(text_blob.strip()) == 0:
        continue
    wc = WordCloud(width=800, height=400, background_color="white").generate(text_blob)
    safe_name = re.sub(r"[^A-Za-z0-9]+", "_", cls)
    wc.to_file(f"output/wordclouds/{safe_name}.png")

In [ ]:
import matplotlib.image as mpimg

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
for i, cls in enumerate(top_classes):
    safe_name = re.sub(r"[^A-Za-z0-9]+", "_", cls)
    path = f"output/wordclouds/{safe_name}.png"
    if os.path.exists(path):
        img = mpimg.imread(path)
        axes[i].imshow(img)
        axes[i].axis("off")
        axes[i].set_title(cls)
plt.tight_layout()
plt.show()

## 7. Feature Extraction

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

count_vectorizer = CountVectorizer(max_features=10000, ngram_range=(1, 2))
tfidf_vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))

count_matrix = count_vectorizer.fit_transform(processed_df["clean_text"])
tfidf_matrix = tfidf_vectorizer.fit_transform(processed_df["clean_text"])

print("CountVectorizer dimensionality:", count_matrix.shape)
print("TfidfVectorizer dimensionality:", tfidf_matrix.shape)
print("Sample Count features:", count_vectorizer.get_feature_names_out()[:10])
print("Sample TF-IDF features:", tfidf_vectorizer.get_feature_names_out()[:10])

## 8. Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X = processed_df["clean_text"]
y = processed_df[LABEL_COLUMN]

if X.shape[0] < 10:
    raise ValueError("Insufficient samples to train a model")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("Train size:", X_train.shape[0])
print("Test size:", X_test.shape[0])

## 9. Model Training and Comparison

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.calibration import CalibratedClassifierCV

vectorizers = {
    "count": CountVectorizer(max_features=10000, ngram_range=(1, 2)),
    "tfidf": TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
}

model_defs = {
    "MultinomialNB": (MultinomialNB(), {"clf__alpha": [0.1, 1.0]}),
    "LogisticRegression": (LogisticRegression(max_iter=200, solver="lbfgs", multi_class="auto"), {"clf__C": [0.5, 1.0]}),
    "LinearSVC": (LinearSVC(max_iter=2000), {"clf__C": [0.5, 1.0]}),
    "RandomForest": (RandomForestClassifier(random_state=RANDOM_STATE), {"clf__n_estimators": [100], "clf__max_depth": [None, 30]})
}

In [ ]:
trained_models = {}

for vec_name, vectorizer in vectorizers.items():
    for model_name, (estimator, param_grid) in model_defs.items():
        pipeline = Pipeline([
            ("vectorizer", vectorizer),
            ("clf", estimator)
        ])
        search = GridSearchCV(pipeline, param_grid, cv=3, scoring="f1_macro", n_jobs=-1)
        search.fit(X_train, y_train)
        key = f"{model_name}_{vec_name}"
        trained_models[key] = search.best_estimator_
        print(key, "best params:", search.best_params_)

## 10. Model Evaluation

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix
)

evaluation_rows = []
classification_reports = {}
confusion_matrices = {}

for key, model in trained_models.items():
    preds = model.predict(X_test)
    accuracy = accuracy_score(y_test, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, preds, average="macro", zero_division=0)
    report = classification_report(y_test, preds, zero_division=0, output_dict=True)
    cm = confusion_matrix(y_test, preds, labels=sorted(y.unique()))
    classification_reports[key] = report
    confusion_matrices[key] = cm
    evaluation_rows.append({
        "model_vectorizer": key,
        "accuracy": accuracy,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1
    })

evaluation_df = pd.DataFrame(evaluation_rows).sort_values("f1_macro", ascending=False).reset_index(drop=True)
evaluation_df

In [ ]:
labels_sorted = sorted(y.unique())
for key, cm in confusion_matrices.items():
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, xticklabels=labels_sorted, yticklabels=labels_sorted, cmap="Blues")
    plt.title(f"Confusion Matrix: {key}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    safe_key = re.sub(r"[^A-Za-z0-9]+", "_", key)
    plt.savefig(f"output/confusion_matrix_{safe_key}.png")
    plt.close()
print("Confusion matrices saved to output/")

## 11. Model Comparison Dashboard

In [ ]:
comparison_df = evaluation_df[["model_vectorizer", "accuracy", "f1_macro"]].sort_values("f1_macro", ascending=False)
comparison_df

In [ ]:
plt.figure(figsize=(12, 6))
melted = comparison_df.melt(id_vars="model_vectorizer", value_vars=["accuracy", "f1_macro"])
sns.barplot(data=melted, x="model_vectorizer", y="value", hue="variable")
plt.xticks(rotation=45, ha="right")
plt.title("Model Comparison: Accuracy vs F1 Macro")
plt.tight_layout()
plt.savefig("output/model_comparison.png")
plt.show()

## 12. TF-IDF vs CountVectorizer Analysis

In [ ]:
best_overall_key = evaluation_df.iloc[0]["model_vectorizer"]
best_model_name = best_overall_key.rsplit("_", 1)[0]

count_key = f"{best_model_name}_count"
tfidf_key = f"{best_model_name}_tfidf"

print("Comparing", count_key, "vs", tfidf_key)
print(evaluation_df[evaluation_df["model_vectorizer"].isin([count_key, tfidf_key])])

count_vec_fitted = trained_models[count_key].named_steps["vectorizer"]
tfidf_vec_fitted = trained_models[tfidf_key].named_steps["vectorizer"]

count_feature_sums = count_vec_fitted.transform(X_train).sum(axis=0).A1
tfidf_feature_sums = tfidf_vec_fitted.transform(X_train).sum(axis=0).A1

top_count_idx = count_feature_sums.argsort()[::-1][:10]
top_tfidf_idx = tfidf_feature_sums.argsort()[::-1][:10]

count_features = count_vec_fitted.get_feature_names_out()
tfidf_features = tfidf_vec_fitted.get_feature_names_out()

print("Top CountVectorizer features by raw count:")
for idx in top_count_idx:
    print(count_features[idx], count_feature_sums[idx])

print("Top TF-IDF features by weighted sum:")
for idx in top_tfidf_idx:
    print(tfidf_features[idx], tfidf_feature_sums[idx])

print("Explanation: TF-IDF downweights terms that appear frequently across many documents, so common but uninformative words receive lower weight compared to raw counts in CountVectorizer.")

## 13. Most Important Words per Class

In [ ]:
best_pipeline = trained_models[best_overall_key]
best_vectorizer = best_pipeline.named_steps["vectorizer"]
best_classifier = best_pipeline.named_steps["clf"]
feature_names = best_vectorizer.get_feature_names_out()

importance_rows = []

if hasattr(best_classifier, "coef_"):
    coefs = best_classifier.coef_
    classes_for_coef = best_classifier.classes_
    for i, cls in enumerate(classes_for_coef):
        row = coefs[i] if coefs.shape[0] > 1 else coefs[0]
        top_idx = row.argsort()[::-1][:10]
        for idx in top_idx:
            importance_rows.append({"class": cls, "word": feature_names[idx], "score": row[idx]})
elif hasattr(best_classifier, "feature_log_prob_"):
    for i, cls in enumerate(best_classifier.classes_):
        row = best_classifier.feature_log_prob_[i]
        top_idx = row.argsort()[::-1][:10]
        for idx in top_idx:
            importance_rows.append({"class": cls, "word": feature_names[idx], "score": row[idx]})
elif hasattr(best_classifier, "feature_importances_"):
    row = best_classifier.feature_importances_
    top_idx = row.argsort()[::-1][:10]
    for idx in top_idx:
        importance_rows.append({"class": "overall", "word": feature_names[idx], "score": row[idx]})

importance_df = pd.DataFrame(importance_rows)
importance_df

## 14. Confidence Scores and Prediction Function

In [ ]:
def predict_text(text):
    cleaned = clean_text(text)
    tokens = word_tokenize(cleaned)
    tokens = [t for t in tokens if t not in STOPWORD_SET and len(t) > 1]
    tokens = [LEMMATIZER.lemmatize(t) for t in tokens]
    final_text = " ".join(tokens)

    pipeline = best_pipeline
    classifier = pipeline.named_steps["clf"]

    if hasattr(classifier, "predict_proba"):
        proba = pipeline.predict_proba([final_text])[0]
        classes = classifier.classes_
    elif hasattr(classifier, "decision_function"):
        scores = pipeline.decision_function([final_text])[0]
        exp_scores = np.exp(scores - np.max(scores))
        proba = exp_scores / exp_scores.sum()
        classes = classifier.classes_
    else:
        pred = pipeline.predict([final_text])[0]
        return {"category": pred, "confidence": 1.0, "top_3": [(pred, 1.0)]}

    top_idx = proba.argsort()[::-1][:3]
    top_3 = [(classes[i], float(proba[i])) for i in top_idx]
    predicted_category = classes[int(np.argmax(proba))]
    confidence = float(np.max(proba))

    return {
        "category": predicted_category,
        "confidence": confidence,
        "top_3": top_3
    }

In [ ]:
example_result = predict_text("Apple launches its newest AI-powered smartphone.")
print(example_result)

## 15. Misclassification Analysis

In [ ]:
best_cm = confusion_matrices[best_overall_key]
normalized_cm = best_cm.astype(float) / best_cm.sum(axis=1, keepdims=True)
np.fill_diagonal(normalized_cm, 0)

confusion_pairs = []
n_labels = len(labels_sorted)
for i in range(n_labels):
    for j in range(n_labels):
        if i != j and normalized_cm[i, j] > 0:
            confusion_pairs.append((labels_sorted[i], labels_sorted[j], normalized_cm[i, j]))

confusion_pairs.sort(key=lambda x: x[2], reverse=True)
top_confusions = confusion_pairs[:10]

confusion_analysis_rows = []
for actual_cls, predicted_cls, rate in top_confusions:
    text_a = " ".join(processed_df[processed_df[LABEL_COLUMN] == actual_cls]["clean_text"])
    text_b = " ".join(processed_df[processed_df[LABEL_COLUMN] == predicted_cls]["clean_text"])
    local_tfidf = TfidfVectorizer(max_features=2000)
    try:
        local_tfidf.fit([text_a, text_b])
        matrix = local_tfidf.transform([text_a, text_b]).toarray()
        feature_names_local = local_tfidf.get_feature_names_out()
        overlap_scores = matrix[0] * matrix[1]
        top_overlap_idx = overlap_scores.argsort()[::-1][:10]
        overlapping_terms = [feature_names_local[i] for i in top_overlap_idx if overlap_scores[i] > 0]
    except ValueError:
        overlapping_terms = []
    confusion_analysis_rows.append({
        "actual_class": actual_cls,
        "predicted_class": predicted_cls,
        "confusion_rate": rate,
        "overlapping_terms": overlapping_terms
    })

confusion_analysis_df = pd.DataFrame(confusion_analysis_rows)
confusion_analysis_df.to_csv("output/misclassification_analysis.csv", index=False)
confusion_analysis_df

## 16. Save Artifacts

In [ ]:
import joblib

os.makedirs("output", exist_ok=True)
joblib.dump(best_pipeline, "output/best_model_pipeline.joblib")
evaluation_df.to_csv("output/evaluation_metrics.csv", index=False)
importance_df.to_csv("output/feature_importance.csv", index=False)

print("Best model saved:", best_overall_key)
print("Artifacts saved to output/ directory")

## 17. Reproducibility and Package Versions

In [ ]:
import sklearn
import nltk as nltk_module
import wordcloud as wordcloud_module

print("pandas", pd.__version__)
print("numpy", np.__version__)
print("scikit-learn", sklearn.__version__)
print("nltk", nltk_module.__version__)
print("matplotlib", plt.matplotlib.__version__)
print("seaborn", sns.__version__)
print("wordcloud", wordcloud_module.__version__)